# Fake News Detection System: AI-Powered News Authenticity Classification
### Machine Learning Final Project
**Dataset:** WELFake (Word Embedding-based and Linguistic Feature Assessment for Fake News Detection)  
**Dataset Source:** IEEE Transactions on Computational Social Systems (2021)  
**Authors:** P. K. Verma, P. Agrawal, I. Amorim, R. Prodan  
**Goal:** Formulate an NLP text classification pipeline to detect fabricated versus authentic news.

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

sns.set_theme(style='whitegrid')
print('All libraries loaded successfully!')

## 1. Dataset Collection and Exploratory Inspection
The WELFake dataset is an authoritative benchmark aggregating 72,134 news records from Kaggle, McIntire, and Reuters.
- Target `label` definition: `0` denotes REAL news, `1` denotes FAKE news.

In [2]:
dataset_path = '../dataset/WELFake_Dataset.csv'
df_raw = pd.read_csv(dataset_path)
print(f'Total records: {df_raw.shape[0]:,}, Features: {df_raw.shape[1]}')
print(df_raw.info())
df_raw.head(3)

## 2. Text Preprocessing & Cleaning
We handle missing fields, remove duplicate records, concatenate the headline (`title`) with body (`text`), and clean HTML/URL noise.

In [3]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = df_raw.dropna(subset=['label']).copy()
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['combined'] = df['title'] + ' ' + df['text']
df = df.drop_duplicates(subset=['combined'])

# Stratified sample for balanced, fast notebook execution
sample_size = 25000
half = sample_size // 2
real_df = df[df['label'] == 0].sample(n=min(half, len(df[df['label'] == 0])), random_state=42)
fake_df = df[df['label'] == 1].sample(n=min(half, len(df[df['label'] == 1])), random_state=42)
df_sampled = pd.concat([real_df, fake_df]).sample(frac=1.0, random_state=42).reset_index(drop=True)
df_sampled['clean_content'] = df_sampled['combined'].apply(clean_text)
print(f'Processed sample size: {len(df_sampled):,}')

## 3. Train-Test Splitting (80/20 Stratified)

In [4]:
X = df_sampled['clean_content']
y = df_sampled['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train set: {len(X_train):,} samples, Test set: {len(X_test):,} samples')

## 4. Model Training & Benchmarking
We test 3 machine learning algorithms embedded inside Scikit-learn Pipelines with TF-IDF:
1. **Logistic Regression**
2. **Multinomial Naive Bayes**
3. **Linear SVM (Calibrated)**

In [5]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)

models = {
    'Logistic Regression': Pipeline([
        ('tfidf', tfidf),
        ('clf', LogisticRegression(max_iter=1000, C=2.0, random_state=42))
    ]),
    'Multinomial Naive Bayes': Pipeline([
        ('tfidf', tfidf),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', tfidf),
        ('clf', CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000), cv=3))
    ])
}

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'pipeline': pipeline, 'accuracy': acc, 'f1': f1}
    print(f'{name:26} -> Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%')

## 5. Model Serialization
Save the best model pipeline for FastAPI integration.

In [6]:
best_name = max(results.keys(), key=lambda k: results[k]['f1'])
print(f'Selected Best Model: {best_name}')
joblib.dump(results[best_name]['pipeline'], '../backend/model/fake_news_pipeline.joblib')
print('Model successfully serialized to backend/model/fake_news_pipeline.joblib')